<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m1_03_categorical_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M1.03 — Using categorical features

In the lab we encoded the categorical columns of the adult census with
`OrdinalEncoder` and `OneHotEncoder`, and combined them with the numerical
columns in a `ColumnTransformer`. In this workbook you do the same **on the
1985 wages survey**, and you will meet categories the model has never seen,
a trap in how the data is ordered, and a question of fairness.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "all_features_accuracy": "342a441fd887344f",
    "n_binary_columns": "4b227777d4dd1fc6",
    "n_categorical": "7902699be42c8a8e",
    "n_columns_if_binary": "b17ef6d19c7a5b1e",
    "n_onehot_columns": "f5ca38f748a1d6ea",
    "n_ones_unknown": "5feceb66ffc86f38",
    "worst_fold": "ef2d127de37b942b"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Re-read the text above this cell and try again."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

As before, each row is a person surveyed in 1985 and we predict whether
they earn **more than $10 an hour**. This time we use **all** the other
columns.

In [ ]:
import pandas as pd

wages = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/cps_85_wages.csv"
)
target = wages["WAGE"] > 10
data = wages.drop(columns="WAGE")
data.head()

## 2. Selecting columns by type

`make_column_selector` picks columns by their type. Columns that are not
numbers hold labels: they are our **categorical** columns.

In [ ]:
from sklearn.compose import make_column_selector

categorical_selector = make_column_selector(dtype_exclude="number")
numerical_selector = make_column_selector(dtype_include="number")

categorical_columns = categorical_selector(data)
numerical_columns = numerical_selector(data)
print("categorical:", categorical_columns)
print("numerical:  ", numerical_columns)

✏️ How many categorical columns are there?

In [ ]:
n_categorical = ____
check("n_categorical", n_categorical)

## 3. One-hot encoding

A model needs numbers. `OneHotEncoder` replaces a categorical column by
**one new column per category**, holding 1 if the person belongs to that
category and 0 otherwise.

Let us look at the categories of `OCCUPATION`:

In [ ]:
data["OCCUPATION"].value_counts()

🔮 **Predict first.** How many columns will `OneHotEncoder` create from
`OCCUPATION` alone? Enter a number.

In [ ]:
my_prediction = ____
commit(my_prediction, list(range(0, 20)))

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
occupation_encoded = encoder.fit_transform(data[["OCCUPATION"]])
pd.DataFrame(
    occupation_encoded, columns=encoder.get_feature_names_out()
).head()

Now all 7 categorical columns at once. Here is how many categories each one
has:

In [ ]:
data[categorical_columns].nunique()

✏️ Using the counts above, how many columns will one-hot encoding of
**all** categorical columns produce? Compute it yourself, then run the
next cell to see if you were right.

In [ ]:
n_onehot_columns = ____
check("n_onehot_columns", n_onehot_columns)

In [ ]:
OneHotEncoder(sparse_output=False).fit_transform(data[categorical_columns]).shape

### ➕ Going further: binary columns

Four columns (`SOUTH`, `SEX`, `UNION`, `MARR`) have only two categories. For
such a column, two one-hot columns are redundant: if `SEX_female` is 0, we
already know `SEX_male` is 1. The option `drop="if_binary"` keeps a single
column for binary features.

✏️ How many columns will we get with `drop="if_binary"`?

In [ ]:
n_columns_if_binary = ____
check("n_columns_if_binary", n_columns_if_binary)

In [ ]:
OneHotEncoder(drop="if_binary", sparse_output=False).fit_transform(
    data[categorical_columns]
).shape

## 4. ➕ Going further: a category the model has never seen

The encoder learns the list of categories during `fit`. What happens when,
later, a new person has an occupation that was not in the training data?

In [ ]:
new_person = pd.DataFrame({"OCCUPATION": ["Student"]})

encoder = OneHotEncoder(sparse_output=False)
encoder.fit(data[["OCCUPATION"]])
try:
    encoder.transform(new_person)
except ValueError as error:
    print("ValueError:", error)

By default the encoder refuses: it raises an error. In production, this
would crash the whole prediction service for one unusual person.

The option `handle_unknown` controls this. ✏️ Set it to the value that
makes the encoder **ignore** unknown categories (as a string; see the error
message above or the
[documentation](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)).

In [ ]:
handle_unknown = ____
encoder = OneHotEncoder(handle_unknown=handle_unknown, sparse_output=False)
encoder.fit(data[["OCCUPATION"]])
pd.DataFrame(encoder.transform(new_person), columns=encoder.get_feature_names_out())

✏️ How many 1s are in the encoded row of our student?

In [ ]:
n_ones_unknown = ____
check("n_ones_unknown", n_ones_unknown)

🧠 **Think.** The model receives a row of zeros for `OCCUPATION`. What does
that mean for its prediction? Is ignoring the unknown category always a
good idea?

## 5. A full pipeline

`make_column_transformer` applies a different preprocessing to each group
of columns: one-hot encoding for categorical columns, scaling for numerical
ones. Then a logistic regression makes the prediction.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

preprocessor = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), categorical_selector),
    (StandardScaler(), numerical_selector),
)
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model

### ➕ Going further: does the order of the rows matter?

Let us cross-validate this model with 5 folds, exactly as in the lab:

In [ ]:
from sklearn.model_selection import cross_validate

scores = cross_validate(model, data, target, cv=5)["test_score"]
scores.round(2)

✏️ One fold is much worse than the others. Which one? Give its number,
counting from **1** to 5.

In [ ]:
worst_fold = ____
check("worst_fold", worst_fold)

With `cv=5`, scikit-learn cuts the data **in order**: the first fold tests
on the first fifth of the rows, the last fold on the last fifth. Let us
look at the share of high wages along the file:

In [ ]:
fifth = pd.Series(range(len(target))) * 5 // len(target) + 1
target.groupby(fifth).mean().round(2).rename_axis("fifth of the file")

The rows are not in random order: the share of high wages changes a lot
from one part of the file to another. The last fold is trained on the first
four fifths (roughly a quarter of high wages) and tested on the last fifth,
where more than half of the people earn more than $10. It is tested on
people who are quite different from the ones it was trained on.

The fix: **shuffle** the rows before cutting them into folds.
`StratifiedKFold(shuffle=True)` does that, while keeping the same share of
each class in every fold.

🔮 **Predict first.** With shuffling, will the fold scores be **"more
spread out"**, **"about as spread out"** or **"less spread out"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["more spread out", "about as spread out", "less spread out"])

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_validate(model, data, target, cv=cv)["test_score"]
print(scores.round(2), f"mean {scores.mean():.2f} ± {scores.std():.2f}")

We use this shuffled `cv` for all the comparisons below.

## 6. Do the categorical columns help?

🔮 **Predict first.** Compared with a model that uses **only the numerical
columns**, will adding the categorical columns make the accuracy
**"lower"**, **"about the same"** or **"higher"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["lower", "about the same", "higher"])

In [ ]:
numerical_model = make_pipeline(StandardScaler(), LogisticRegression())
numerical_scores = cross_validate(
    numerical_model, data[numerical_columns], target, cv=cv
)["test_score"]
print(f"numerical only : {numerical_scores.mean():.2f} ± {numerical_scores.std():.2f}")
print(f"all features   : {scores.mean():.2f} ± {scores.std():.2f}")

✏️ Mean accuracy with all features, rounded to 2 decimals:

In [ ]:
all_features_accuracy = ____
check("all_features_accuracy", all_features_accuracy)

## 7. ➕ Going further: ordinal instead of one-hot

`OrdinalEncoder` replaces each category by a **single number** (0, 1, 2…)
in alphabetical order. For `OCCUPATION` that gives:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal = OrdinalEncoder().fit(data[["OCCUPATION"]])
dict(zip(ordinal.categories_[0], range(len(ordinal.categories_[0]))))

🧠 **Think.** For a linear model, `Service` (5) now counts "five times more"
than `Management` (1). Does that order mean anything?

🔮 **Predict first.** Will the ordinal version of the pipeline score
**"much lower"**, **"about the same"** or **"much higher"** than the one-hot
version?

In [ ]:
my_prediction = ____
commit(my_prediction, ["much lower", "about the same", "much higher"])

In [ ]:
ordinal_model = make_pipeline(
    make_column_transformer(
        (OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_selector),
        (StandardScaler(), numerical_selector),
    ),
    LogisticRegression(max_iter=1000),
)
ordinal_scores = cross_validate(ordinal_model, data, target, cv=cv)["test_score"]
print(f"one-hot : {scores.mean():.2f} ± {scores.std():.2f}")
print(f"ordinal : {ordinal_scores.mean():.2f} ± {ordinal_scores.std():.2f}")

The difference is small here. ✏️ One reason: for a column with only **two**
categories, ordinal encoding (0/1) and one-hot encoding carry exactly the
same information. How many of our 7 categorical columns are binary?

In [ ]:
n_binary_columns = ____
check("n_binary_columns", n_binary_columns)

## 8. ➕ Going further: should the model use `SEX` and `RACE`?

In workbook M1.01 we saw that the share of high wages differs between men
and women. A model trained on this data can learn that difference and
reproduce it.

🔮 **Predict first.** If we remove `SEX` and `RACE` from the features, will
the accuracy **"drop a lot"**, **"change only a little"** or **"go up a
lot"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["drop a lot", "change only a little", "go up a lot"])

In [ ]:
data_without = data.drop(columns=["SEX", "RACE"])
without_scores = cross_validate(model, data_without, target, cv=cv)["test_score"]
print(f"all features       : {scores.mean():.2f} ± {scores.std():.2f}")
print(f"without SEX & RACE : {without_scores.mean():.2f} ± {without_scores.std():.2f}")

🧠 **Think.**

1. Is the difference larger than the fold-to-fold variation (the `±`)?
2. Removing `SEX` does not guarantee the model treats men and women
   equally: other columns (`OCCUPATION`, `SECTOR`…) may be correlated with
   it. Can you think of an example?
3. If this model were used to propose salaries, would you keep these
   columns? Why?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - how one-hot encoding works and how many columns it creates here;
   - what `handle_unknown="ignore"` does and when it is risky;
   - why one fold scored so badly and how shuffling fixed it;
   - whether this model should use `SEX` and `RACE`.